# 00 Cellpose segmentation and single-cell crops

Segments the cells in the original images with a trained Cellpose model, checks every mask with morphological
criteria and saves each accepted cell as a square crop with the original intensities. Used for the z-stack, the
Andor Zyla flow and the static Phantom datasets. (The Phantom flow and APD crops come from the tracking analysis of
the high-speed recordings.)

1. Cellpose on a normalized copy of each image (1st and 99th percentile).
2. Shape descriptors of every mask after one binary closing and hole filling: area, circularity, aspect ratio,
   solidity, eccentricity, normalized radial standard deviation, convexity ratio.
3. Square crop around every accepted mask: side = longer bounding-box side + 2 × `MARGIN_PX`, centred on the box.
   Crops that reach beyond the image are padded with the median of the image border.
4. `summary.csv` in `OUTPUT_ROOT` with one row per mask (accepted and rejected), the shape descriptors, the bounding
   box and the paths. The classification notebooks use it for the edge check of the Andor Zyla flow crops.

Run the notebook once per input folder (cell line, imaging mode and day). Already processed images are skipped, so
an interrupted run can be continued.

Thesis: Methods, Section "Cell segmentation with Cellpose" (Table `tab:shape_filter`, Figure
`fig:segmentation_example`).

## 1. Imports

In [ ]:
from pathlib import Path
import csv
import gc
import inspect

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile as tiff
import torch
from cellpose import core, models
from IPython.display import display
from scipy import ndimage as ndi
from skimage.measure import label, perimeter_crofton, regionprops
from skimage.morphology import convex_hull_image
from skimage.segmentation import find_boundaries

## 2. Settings
Only this cell needs to be changed. `PRESET` selects the segmentation parameters and the morphological filter of one
dataset. The input and output folders and the Cellpose model are set per run.

In [ ]:
# ---- input and output
INPUT_ROOT = Path('/Volumes/Extreme SSD/Masterarbeit/cellpose/snapshot_phantom/201026/HeLa/BF')   # searched recursively
OUTPUT_ROOT = Path('/Volumes/Extreme SSD/Masterarbeit/cellpose/snapshot_phantom/single_crops/HeLa/BF/201026')
PRETRAINED_MODEL = '/Volumes/Extreme SSD/Masterarbeit/cellpose/snapshot_phantom/141026/HeLa/BF/models/HeLa_BF'
PRESET = 'phantom_static'          # 'zstack', 'flow_zyla' or 'phantom_static'

# ---- segmentation parameters and morphological filter per dataset
PRESETS = {
    # z-stack and Andor Zyla flow (0.325 µm/px): strict filter for compact, nearly round masks
    'zstack': dict(diameter=50, cellprob_threshold=-1.5, flow_threshold=0.4, min_size=1000,
                   min_area=1500, min_circularity=0.70, max_aspect_ratio=1.50, min_solidity=0.82,
                   max_eccentricity=0.88, max_radial_std_norm=0.20, min_convexity_ratio=0.80),
    # static Phantom (0.925 µm/px)
    'phantom_static': dict(diameter=18, cellprob_threshold=0.0, flow_threshold=0.4, min_size=15,
                           min_area=50, min_circularity=0.70, max_aspect_ratio=1.50, min_solidity=0.82,
                           max_eccentricity=0.88, max_radial_std_norm=0.20, min_convexity_ratio=0.80),
}
PRESETS['flow_zyla'] = PRESETS['zstack']      # same camera, pixel size, models and parameters as the z-stack
P = PRESETS[PRESET]

USE_GPU = True
INVERT = False                 # True: invert the image before Cellpose
MARGIN_PX = 3                  # background margin around the bounding box, pixels per side
MODEL_RELOAD_EVERY = 15        # reload the model every N images to limit the memory use
N_PREVIEW = 3                  # images shown with their masks before the main run (0: no preview)
IMAGE_EXTENSIONS = {'.tif', '.tiff', '.png', '.jpg', '.jpeg'}

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Preset {PRESET!r}:', P)

## 3. Images

In [ ]:
files = sorted(p for p in INPUT_ROOT.rglob('*')
               if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS and not p.name.startswith('._'))
print(f'{len(files)} images found in {INPUT_ROOT}')

## 4. Image functions and crop extraction

In [ ]:
def load_image(path):
    img = np.squeeze(np.asarray(tiff.imread(path)))
    return img.mean(axis=-1) if img.ndim == 3 else img


def normalize_percentile(img, p_low=1, p_high=99):
    arr = np.asarray(img).astype(np.float32)
    lo, hi = np.percentile(arr, [p_low, p_high])
    if hi <= lo:
        lo, hi = arr.min(), arr.max()
    if hi <= lo:
        return np.zeros_like(arr)
    return np.clip((arr - lo) / (hi - lo), 0.0, 1.0).astype(np.float32)


def to_cellpose_input(img):
    # normalized copy for Cellpose. The crops are cut from the original image.
    arr = normalize_percentile(img)
    return (1.0 - arr if INVERT else arr).astype(np.float32)


def estimate_background_value(img):
    # median of a border of one tenth of the shorter image side
    arr = np.asarray(img)
    h, w = arr.shape
    bw = max(1, min(h, w) // 10)
    border = np.concatenate([arr[:bw, :].ravel(), arr[-bw:, :].ravel(), arr[:, :bw].ravel(), arr[:, -bw:].ravel()])
    return np.median(border).astype(arr.dtype)


def crop_square_from_mask(image, mask, margin_px=MARGIN_PX):
    # square crop centred on the bounding box of the mask, padded with the background value at the image border
    img = np.asarray(image)
    ys, xs = np.where(np.asarray(mask).astype(bool))
    if len(ys) == 0:
        return None, None
    y0, y1 = int(ys.min()), int(ys.max()) + 1
    x0, x1 = int(xs.min()), int(xs.max()) + 1
    bbox_h, bbox_w = y1 - y0, x1 - x0
    side = int(max(bbox_h, bbox_w) + 2 * margin_px)
    cy, cx = (y0 + y1) / 2.0, (x0 + x1) / 2.0
    y_start, x_start = int(round(cy - side / 2.0)), int(round(cx - side / 2.0))
    y_end, x_end = y_start + side, x_start + side
    H, W = img.shape[:2]
    pad_top, pad_left = max(0, -y_start), max(0, -x_start)
    pad_bottom, pad_right = max(0, y_end - H), max(0, x_end - W)
    if pad_top or pad_bottom or pad_left or pad_right:
        padded = np.pad(img, ((pad_top, pad_bottom), (pad_left, pad_right)), mode='constant',
                        constant_values=estimate_background_value(img))
        crop = padded[y_start + pad_top:y_end + pad_top, x_start + pad_left:x_end + pad_left]
    else:
        crop = img[y_start:y_end, x_start:x_end]
    return crop, {'bbox_y0': y0, 'bbox_x0': x0, 'bbox_y1': y1, 'bbox_x1': x1, 'bbox_h': bbox_h, 'bbox_w': bbox_w,
                  'final_side_px': int(crop.shape[0])}

## 5. Cellpose model

In [ ]:
def load_model():
    use_gpu = bool(USE_GPU and core.use_gpu())
    if not Path(PRETRAINED_MODEL).is_file():
        raise FileNotFoundError(f'Cellpose model not found: {PRETRAINED_MODEL}')
    return models.CellposeModel(gpu=use_gpu, pretrained_model=str(PRETRAINED_MODEL)), use_gpu


def run_cellpose(img_raw):
    kwargs = dict(diameter=P['diameter'], flow_threshold=P['flow_threshold'],
                  cellprob_threshold=P['cellprob_threshold'], min_size=P['min_size'], normalize=True)
    accepted = set(inspect.signature(model.eval).parameters)
    if 'channels' in accepted:
        kwargs['channels'] = [0, 0]
    elif 'channel_axis' in accepted:
        kwargs['channel_axis'] = None
    result = model.eval(to_cellpose_input(img_raw), **kwargs)
    return np.asarray(result[0] if isinstance(result, (list, tuple)) else result)


def flush_gpu_memory():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        torch.mps.empty_cache()
    gc.collect()


model, use_gpu = load_model()
print(f'Model loaded: {PRETRAINED_MODEL} | GPU/MPS: {use_gpu}')

## 6. Morphological filter

In [ ]:
def compute_radial_std_norm(mask, centroid_r, centroid_c):
    ys, xs = np.where(find_boundaries(mask, mode='inner'))
    if len(ys) < 5:
        return 0.0
    d = np.sqrt((ys - centroid_r) ** 2 + (xs - centroid_c) ** 2)
    return float(d.std() / d.mean()) if d.mean() > 1e-6 else 0.0


def compute_convexity_ratio(mask, perimeter_val):
    if perimeter_val <= 0:
        return 1.0
    try:
        convex_perim = float(perimeter_crofton(convex_hull_image(mask), directions=4))
        return float(convex_perim / perimeter_val) if convex_perim > 0 else 1.0
    except Exception:
        return 1.0


def compute_shape_metrics(mask):
    # one binary closing and hole filling before the descriptors are calculated
    m = ndi.binary_fill_holes(ndi.binary_closing(mask.astype(bool), iterations=1))
    props = regionprops(label(m.astype(np.uint8)))
    if not props:
        return None
    region = max(props, key=lambda r: r.area)
    area = float(region.area)
    try:
        perim = float(perimeter_crofton(m, directions=4))
    except Exception:
        perim = float(region.perimeter)
    minr, minc, maxr, maxc = region.bbox
    h, w = maxr - minr, maxc - minc
    cr, cc = region.centroid
    return {'area': area, 'perimeter': perim,
            'circularity': 4.0 * np.pi * area / perim ** 2 if perim > 0 else 0.0,
            'aspect_ratio': max(h, w) / min(h, w) if min(h, w) > 0 else np.inf,
            'solidity': float(region.solidity) if region.solidity else 0.0,
            'eccentricity': float(region.eccentricity),
            'radial_std_norm': compute_radial_std_norm(m, cr, cc),
            'convexity_ratio': compute_convexity_ratio(m, perim),
            'bbox_y0': int(minr), 'bbox_x0': int(minc), 'bbox_y1': int(maxr), 'bbox_x1': int(maxc)}


def check_mask(mask):
    # (accepted, descriptors, reason)
    if mask is None or mask.sum() == 0:
        return False, None, 'empty mask'
    m = compute_shape_metrics(mask)
    if m is None:
        return False, None, 'no regionprops'
    if m['area'] < P['min_area']:
        return False, m, 'too small'
    if m['circularity'] < P['min_circularity']:
        return False, m, 'low circularity'
    if m['aspect_ratio'] > P['max_aspect_ratio']:
        return False, m, 'bad aspect ratio'
    if m['solidity'] < P['min_solidity']:
        return False, m, 'low solidity'
    if m['eccentricity'] > P['max_eccentricity']:
        return False, m, 'too elongated'
    if m['radial_std_norm'] > P['max_radial_std_norm']:
        return False, m, 'high radial irregularity'
    if m['convexity_ratio'] < P['min_convexity_ratio']:
        return False, m, 'low convexity'
    return True, m, 'accepted'


def filter_masks(masks):
    accepted, rejected = [], []
    for lid in sorted(int(x) for x in np.unique(masks) if int(x) != 0):
        obj = masks == lid
        ok, metrics, reason = check_mask(obj)
        (accepted if ok else rejected).append({'label_id': lid, 'mask': obj, 'metrics': metrics, 'reason': reason})
    return accepted, rejected

## 7. Preview
Original image, Cellpose masks and the accepted (green) and rejected (red) masks for the first `N_PREVIEW` images.

In [ ]:
for path in files[:N_PREVIEW]:
    img_raw = load_image(path)
    masks = run_cellpose(img_raw)
    accepted, rejected = filter_masks(masks)
    base = normalize_percentile(img_raw)
    rgb = np.dstack([base, base, base])
    for items, colour in ((accepted, [0, 1, 0]), (rejected, [1, 0, 0])):
        for item in items:
            rgb[find_boundaries(item['mask'], mode='outer')] = colour
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(base, cmap='gray', vmin=0, vmax=1); axes[0].set_title(path.name)
    axes[1].imshow(masks, cmap='nipy_spectral'); axes[1].set_title(f'Cellpose masks, n = {int(masks.max())}')
    axes[2].imshow(rgb); axes[2].set_title(f'accepted = {len(accepted)}, rejected = {len(rejected)}')
    for ax in axes:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## 8. All images: crops and summary.csv

In [ ]:
def process_one_image(path):
    # None if the image was processed before (its crops exist), otherwise one row per mask
    rel = path.relative_to(INPUT_ROOT)
    out_dir = OUTPUT_ROOT / rel.parent
    out_dir.mkdir(parents=True, exist_ok=True)
    if list(out_dir.glob(f'{path.stem}_cell*.tif')):
        return None
    img_raw = load_image(path)
    masks = run_cellpose(img_raw)
    accepted, rejected = filter_masks(masks)
    def row(output_path, status, reason, label_id):
        # same column order as in earlier runs, so that an existing summary.csv can be continued
        return {'input_path': str(path), 'relative_path': str(rel), 'output_path': output_path, 'status': status,
                'reason': reason, 'label_id': label_id, 'was_inverted': INVERT, 'n_cells_found': int(masks.max()),
                'n_accepted': len(accepted), 'n_rejected': len(rejected)}

    rows = []
    for item in accepted:
        crop, crop_info = crop_square_from_mask(img_raw, item['mask'])
        if crop is None:
            continue
        out_path = out_dir / f"{path.stem}_cell{item['label_id']:04d}.tif"
        tiff.imwrite(out_path, crop)
        rows.append({**row(str(out_path), 'accepted', 'accepted', item['label_id']), **item['metrics'], **crop_info})
    for item in rejected:
        rows.append({**row('', 'rejected', item['reason'], item['label_id']), **(item['metrics'] or {})})
    return rows


csv_path = OUTPUT_ROOT / 'summary.csv'
header_written = csv_path.exists()          # append when an interrupted run is continued
n = dict(skipped=0, accepted=0, rejected=0, errors=0)
for i, path in enumerate(files):
    try:
        rows = process_one_image(path)
        if rows is None:
            n['skipped'] += 1
        elif rows:
            pd.DataFrame(rows).to_csv(csv_path, mode='a', header=not header_written, index=False)
            header_written = True
            n['accepted'] += sum(r['status'] == 'accepted' for r in rows)
            n['rejected'] += sum(r['status'] == 'rejected' for r in rows)
    except Exception as e:
        print(f'{path.name}: ERROR {e}')
        n['errors'] += 1
        with open(csv_path, 'a', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=['input_path', 'status', 'reason'])
            if not header_written:
                writer.writeheader()
                header_written = True
            writer.writerow({'input_path': str(path), 'status': 'error', 'reason': str(e)})
    flush_gpu_memory()
    if (i + 1) % MODEL_RELOAD_EVERY == 0 and (i + 1) < len(files):
        del model
        flush_gpu_memory()
        model, _ = load_model()
print(f'{len(files)} images:', n, '| summary:', csv_path)

## 9. Summary

In [ ]:
summary = pd.read_csv(csv_path)
print(summary.status.value_counts().to_dict())
display(summary.loc[summary.status.eq('rejected'), 'reason'].value_counts())